# Sistema Inteligente de Monitoramento - Missão Espacial
## Global Solution - FIAP 2026

Carlos Henrique De Godoy Santos - RM: 569735
Natália Souza Carvalho - RM: 569068

In [ ]:
# Importação das bibliotecas
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
# Leitura dos dados 
df = pd.read_csv('../data/dados.csv')

# Tipagem da coluna de data e hora
df['data_hora'] = pd.to_datetime(df['data_hora'])

# Total de registro
print(f'Total de registros: {len(df)}')

Total de registros: 96


,data_hora,suporte_vida,energia,comunicacao,habitat,laboratorio,armazenamento,geracao_solar_kwh,geracao_eolica_kwh,consumo_kwh,reserva_energia_pct,temperatura_interna_c,temperatura_externa_c,radiacao_uv,qualidade_comunicacao_pct,velocidade_vento_kmh,evento,severidade
0,2026-06-01 00:00:00,1,1,1,1,1,1,30.5,12.0,35.0,82,22.1,-45.3,3.2,95,18.4,Sistema inicializado com sucesso,normal
1,2026-06-01 01:00:00,1,1,1,1,1,1,28.3,11.5,34.2,80,22.0,-46.1,3.1,94,17.9,Leitura de sensores nominal,normal
2,2026-06-01 02:00:00,1,1,1,1,1,1,25.1,10.8,33.8,78,21.8,-47.0,3.3,93,16.5,Rotina de verificacao concluida,normal
3,2026-06-01 03:00:00,1,1,1,1,1,1,22.0,10.2,32.5,76,21.5,-48.2,3.5,92,15.8,Ciclo de manutencao preventiva,normal
4,2026-06-01 04:00:00,1,1,1,1,0,1,18.5,9.8,31.0,73,21.3,-49.5,3.8,90,14.2,Laboratorio desligado para economia,alerta
5,2026-06-01 05:00:00,1,1,1,1,0,1,15.2,9.5,28.5,70,21.0,-50.1,4.0,88,13.5,Modo economia ativado,alerta
6,2026-06-01 06:00:00,1,1,1,1,0,1,20.8,10.0,29.0,68,21.2,-49.8,4.2,87,14.8,Geracao solar em recuperacao,alerta
7,2026-06-01 07:00:00,1,1,1,1,1,1,32.0,11.2,36.5,66,22.3,-48.0,4.5,89,16.0,Laboratorio reativado,normal
8,2026-06-01 08:00:00,1,1,1,1,1,1,38.5,12.5,40.2,64,22.8,-46.5,5.0,91,17.5,Pico de geracao solar matinal,normal
9,2026-06-01 09:00:00,1,1,1,1,1,1,42.0,13.0,42.0,63,23.2,-45.0,5.5,92,18.2,Consumo elevado - experimentos,normal


## 1. Status dos Módulos Críticos (Binário)

In [ ]:
# Modulos criticos binários
modulos = ['suporte_vida', 'energia', 'comunicacao', 'habitat', 'laboratorio', 'armazenamento']

# Dicionário de status dos modulos
status_modulos = {}
for mod in modulos:
    total_falhas = (df[mod] == 0).sum()

    if total_falhas > 5:
        status = 'CRITICO'
    elif total_falhas > 0:
        status = 'ALERTA'
    else:
        status = 'NORMAL'
    

    status_modulos[mod] = {
        'operacional_pct': (df[mod] == 1).mean() * 100,
        'total_falhas': total_falhas,
        'status': status
        }
        
print('=' * 60)
print(f'{"MODULO":<18} {"OPERACIONAL %":<16} {"FALHAS":<10} {"STATUS"}')
print('=' * 60)
for mod, info in status_modulos.items():
    print(f'{mod:<18} {info["operacional_pct"]:>10.1f}%     {info["total_falhas"]:>4}      {info["status"]}')
print('=' * 60)

MODULO             OPERACIONAL %    FALHAS     STATUS
suporte_vida            100.0%        0      NORMAL
energia                  96.9%        3      ALERTA
comunicacao              94.8%        5      ALERTA
habitat                  97.9%        2      ALERTA
laboratorio              46.9%       51      CRITICO
armazenamento            96.9%        3      ALERTA


## 2. Organização dos Dados em Estruturas

In [ ]:
# LISTA - Serie temporal de geracao e consumo
lista_geracao_solar = df['geracao_solar_kwh'].tolist()
lista_consumo = df['consumo_kwh'].tolist()
lista_reserva = df['reserva_energia_pct'].tolist()

print('LISTA - Ultimas 6 leituras de geracao solar:', lista_geracao_solar[-6:])
print('LISTA - Ultimas 6 leituras de consumo:', lista_consumo[-6:])
print()

LISTA - Ultimas 6 leituras de geracao solar: [18.0, 10.0, 3.0, 1.0, 0.5, 0.2]
LISTA - Ultimas 6 leituras de consumo: [28.0, 26.0, 24.0, 23.0, 22.0, 21.5]



In [ ]:
# FILA - Alertas pendentes por ordem de chegada
fila_alertas = []

alertas_df = df[df['severidade'].isin(['alerta', 'critico'])].copy()
for i, row in alertas_df.iterrows():
    # Enfileirar: adiciona no final da fila
    fila_alertas.append({
        'timestamp': str(row['timestamp']),
        'evento': row['evento'],
        'severidade': row['severidade']
    })

print(f'FILA DE ALERTAS - Total pendentes: {len(fila_alertas)}')
print('Proximo alerta a processar (frente da fila):', fila_alertas[0] if fila_alertas else 'Nenhum')
print()

# Remove o inicio da Fila
alerta_processado = fila_alertas.pop(0)
print(f'Alerta processado: {alerta_processado["evento"]}')
print(f'Alertas restantes na fila: {len(fila_alertas)}')

FILA DE ALERTAS - Total pendentes: 29
Proximo alerta a processar: {'data_hora': '2026-06-01 04:00:00', 'evento': 'Laboratorio desligado para economia', 'severidade': 'alerta'}



In [ ]:
# PILHA - Ultimos eventos criticos analisados
pilha_criticos = []

criticos_df = df[df['severidade'] == 'critico']
for i , row in criticos_df.iterrows():
    pilha_criticos.append({
        'data_hora': str(row['data_hora']),
        'evento': row['evento']
    })

print(f'PILHA DE EVENTOS CRITICOS - Total: {len(pilha_criticos)}')
print('Ultimo evento critico (topo da pilha):', pilha_criticos[-1] if pilha_criticos else 'Nenhum')
print()

PILHA DE EVENTOS CRITICOS - Total: 11
Ultimo evento critico (topo da pilha): {'data_hora': '2026-06-04 04:00:00', 'evento': 'Tentativa de reconexao via canal secundario'}



In [23]:
# HIERARQUIA - Representacao da missao
hierarquia_missao = {
    'energia': {
        'solar': df['geracao_solar_kwh'].mean(),
        'eolica': df['geracao_eolica_kwh'].mean(),
        'baterias_reserva_pct': df['reserva_energia_pct'].mean()
    },
    'habitat': {
        'oxigenio': status_modulos['suporte_vida']['status'],
        'temperatura_media_c': df['temperatura_interna_c'].mean(),
        'comunicacao': status_modulos['comunicacao']['status']
    }
}

print('HIERARQUIA DA MISSAO:')
for sistema, sub in hierarquia_missao.items():
    print(f'  {sistema.upper()}:')
    for chave, valor in sub.items():
        if isinstance(valor, float):
            print(f'    {chave}: {valor:.2f}')
        else:
            print(f'    {chave}: {valor}')
print()

HIERARQUIA DA MISSAO:
  ENERGIA:
    solar: 21.90
    eolica: 10.57
    baterias_reserva_pct: 52.62
  HABITAT:
    oxigenio: NORMAL
    temperatura_media_c: 21.56
    comunicacao: ALERTA



In [24]:
# MATRIZ - Leituras por horario e variavel (primeiras 24h)
variaveis_matriz = ['geracao_solar_kwh', 'geracao_eolica_kwh', 'consumo_kwh', 'reserva_energia_pct', 'temperatura_interna_c', 'radiacao_uv']
matriz = df[variaveis_matriz].head(24).values.tolist()

print(f'MATRIZ (24 horarios x {len(variaveis_matriz)} variaveis):')
print(f'Colunas: {variaveis_matriz}')
print(f'Dimensoes: {len(matriz)} linhas x {len(matriz[0])} colunas')
print('Primeiras 3 linhas:')
for i, linha in enumerate(matriz[:3]):
    print(f'  Hora {i:02d}: {[round(v, 1) for v in linha]}')

MATRIZ (24 horarios x 6 variaveis):
Colunas: ['geracao_solar_kwh', 'geracao_eolica_kwh', 'consumo_kwh', 'reserva_energia_pct', 'temperatura_interna_c', 'radiacao_uv']
Dimensoes: 24 linhas x 6 colunas
Primeiras 3 linhas:
  Hora 00: [30.5, 12.0, 35.0, 82.0, 22.1, 3.2]
  Hora 01: [28.3, 11.5, 34.2, 80.0, 22.0, 3.1]
  Hora 02: [25.1, 10.8, 33.8, 78.0, 21.8, 3.3]


## 3. Regras Lógicas de Diagnóstico

In [ ]:
def diagnosticar_registro(row):
    alertas = []
    nivel = 'NORMAL'
    
    # Suporte a vida comprometido OU energia falhou com reserva critica
    if row['suporte_vida'] == 0 or (row['energia'] == 0 and row['reserva_energia_pct'] < 30):
        nivel = 'CRITICO'
        alertas.append('Sistemas vitais comprometidos')
    
    # Comunicacao offline E qualidade abaixo do minimo
    if row['comunicacao'] == 0 and not (row['qualidade_comunicacao_pct'] > 20):
        nivel = 'CRITICO'
        alertas.append('Isolamento total de comunicacao')
    elif row['comunicacao'] == 0 and row['qualidade_comunicacao_pct'] <= 50:
        if nivel != 'CRITICO':
            nivel = 'ALERTA'
        alertas.append('Comunicacao instavel')
    
    # Reserva energetica baixa OU radiacao elevada
    if row['reserva_energia_pct'] < 40 or row['radiacao_uv'] > 7.0:
        if nivel == 'NORMAL':
            nivel = 'ALERTA'
        alertas.append('Energia baixa' if row['reserva_energia_pct'] < 40 else 'Radiacao elevada')
    
    # Habitat offline E temperatura fora da faixa segura
    if row['habitat'] == 0 and not (18.0 <= row['temperatura_interna_c'] <= 25.0):
        nivel = 'CRITICO'
        alertas.append('Habitat critico - temperatura fora da faixa segura')
    elif row['habitat'] == 0:
        if nivel == 'NORMAL':
            nivel = 'ALERTA'
        alertas.append('Modulo habitat offline')
    
    return nivel, alertas


# Aplicar diagnostico a todos os registro s
diagnosticos = df.apply(diagnosticar_registro, axis=1)
df['nivel_diagnostico'] = [d[0] for d in diagnosticos]
df['alertas_gerados'] = [d[1] for d in diagnosticos]

print('DISTRIBUICAO DE DIAGNOSTICOS:')
print(df['nivel_diagnostico'].value_counts())
print()

DISTRIBUICAO DE DIAGNOSTICOS:
nivel_diagnostico
NORMAL     74
ALERTA     19
CRITICO     3
Name: count, dtype: int64



## 4. Alertas Automáticos

In [ ]:
def gerar_recomendacao(nivel, alertas, row):
    recomendacoes = []
    
    if 'Sistemas vitais comprometidos' in alertas:
        recomendacoes.append('[CRITICA] Prioridade maxima: restaurar suporte a vida e energia')
    
    if 'Isolamento total de comunicacao' in alertas:
        recomendacoes.append('[CRITICA] Ativar canal de emergencia e protocolo de reconexao')
    
    if 'Energia baixa' in alertas:
        recomendacoes.append('[ALTA] Desligar sistemas nao essenciais (laboratorio, armazenamento)')
        recomendacoes.append('[ALTA] Redirecionar energia para habitat e suporte a vida')
    
    if 'Radiacao elevada' in alertas:
        recomendacoes.append('[ALTA] Ativar protocolo de protecao radiologica')
        recomendacoes.append('[MEDIA] Recolher equipamentos externos')
    
    if 'Habitat critico' in ' '.join(alertas):
        recomendacoes.append('[CRITICA] Restaurar aquecimento imediatamente')
        recomendacoes.append('[ALTA] Tripulacao deve usar trajes termicos')
    
    if 'Comunicacao instavel' in alertas:
        recomendacoes.append('[MEDIA] Tentar reconexao via canal secundario')
    
    if not recomendacoes:
        recomendacoes.append('[INFO] Sistema operando dentro dos parametros normais')
    
    return recomendacoes


# Exibir alertas criticos e suas recomendacoes
print('=' * 70)
print('ALERTAS AUTOMATICOS DO SISTEMA')
print('=' * 70)

registros_criticos = df[df['nivel_diagnostico'] == 'CRITICO'].head(10)
for inc, row in registros_criticos.iterrows():
    print(f'\n[{row["data_hora"]}] NIVEL: {row["nivel_diagnostico"]}')
    print(f'  Evento: {row["evento"]}')
    print(f'  Alertas: {", ".join(row["alertas_gerados"])}')
    recomendacoes = gerar_recomendacao(row['nivel_diagnostico'], row['alertas_gerados'], row)
    print('  Recomendacoes:')
    for rec in recomendacoes:
        print(f'    -> {rec}')

ALERTAS AUTOMATICOS DO SISTEMA

[2026-06-03 05:00:00] NIVEL: CRITICO
  Evento: Habitat critico - temperatura interna caindo
  Alertas: Habitat critico - temperatura fora da faixa segura
  Recomendacoes:
    -> [CRITICA] Restaurar aquecimento imediatamente
    -> [ALTA] Tripulacao deve usar trajes termicos

[2026-06-04 03:00:00] NIVEL: CRITICO
  Evento: Comunicacao critica - isolamento da base
  Alertas: Isolamento total de comunicacao
  Recomendacoes:
    -> [CRITICA] Ativar canal de emergencia e protocolo de reconexao

[2026-06-04 04:00:00] NIVEL: CRITICO
  Evento: Tentativa de reconexao via canal secundario
  Alertas: Isolamento total de comunicacao
  Recomendacoes:
    -> [CRITICA] Ativar canal de emergencia e protocolo de reconexao



## 5. Análise e Previsão de Dados (Regressão Linear Simples)

In [ ]:
# Ultimas 24 leituras de reserva energetica
ultimas_24 = df['reserva_energia_pct'].tail(24).tolist()

# X = horas (0 a 23), y = reserva
X = np.array(range(len(ultimas_24))).reshape(-1, 1)
y = np.array(ultimas_24)

# Treinar modelo de regressao linear
modelo = LinearRegression()
modelo.fit(X, y)

# Coeficientes: y = a + b*x
a = modelo.intercept_
b = modelo.coef_[0]

print('PREVISAO DE RESERVA ENERGETICA (Regressao Linear - sklearn)')
print(f'Equacao: reserva = {a:.2f} + ({b:.4f}) * hora')
print(f'Tendencia: {"SUBINDO" if b > 0 else "CAINDO"} ({b:.4f}% por hora)')
print()

# Previsao para as proximas 6 horas
horas_futuras = np.array(range(24, 30)).reshape(-1, 1)
previsoes = modelo.predict(horas_futuras)

print('Previsao para as proximas 6 horas:')
for i, prev in enumerate(previsoes):
    print(f'  Hora +{i + 1}: {prev:.1f}%')

print()

# Decisao baseada na previsao
previsao_6h = previsoes[-1]
if previsao_6h < 30:
    print(f'>>> ALERTA: Reserva prevista em {previsao_6h:.1f}% em 6h. Ativar modo economia AGORA.')
elif previsao_6h < 50:
    print(f'>>> ATENCAO: Reserva prevista em {previsao_6h:.1f}% em 6h. Monitorar consumo.')
else:
    print(f'>>> OK: Reserva prevista em {previsao_6h:.1f}% em 6h. Niveis adequados.')

PREVISAO DE RESERVA ENERGETICA
Equacao: reserva = 50.21 + (0.5400) * hora
Tendencia: SUBINDO (0.5400% por hora)

Previsao para as proximas 6 horas:
  Hora +1: 63.2%
  Hora +2: 63.7%
  Hora +3: 64.2%
  Hora +4: 64.8%
  Hora +5: 65.3%
  Hora +6: 65.9%

>>> OK: Reserva prevista em 65.9% em 6h. Niveis adequados.


## 6. Detecção de Inconsistência nos Dados

In [ ]:
print('VERIFICACAO DE INCONSISTÊNCIAS:')
print('-' * 50)

inconsistencias = []
for idx, row in df.iterrows():
    # Radiacao > 6 será considerada uma inconsistência
    if row['radiacao_uv'] > 6.0 and row['severidade'] == 'normal':
        inconsistencias.append({
            'data_hora': str(row['data_hora']),
            'radiacao': row['radiacao_uv'],
            'severidade_registrada': row['severidade'],
            'evento': row['evento']
        })

if inconsistencias:
    print(f'Encontradas {len(inconsistencias)} inconsistência(s):')
    for inc in inconsistencias:
        print(f'  [{inc["data_hora"]}] Radiacao={inc["radiacao"]} mas severidade="{inc["severidade_registrada"]}"')
        print(f'    Evento: {inc["evento"]}')
    print('\nRECOMENDACAO: Recalibrar sensor e revisar thresholds de alerta.')
else:
    print('Nenhuma inconsistencia detectada.')

VERIFICACAO DE INCONSISTENCIAS:
--------------------------------------------------
Encontradas 9 inconsistencia(s):
  [2026-06-01 11:00:00] Radiacao=6.5 mas severidade="normal"
    Evento: Geracao maxima do periodo
  [2026-06-01 14:00:00] Radiacao=6.8 mas severidade="normal"
    Evento: Comunicacao restabelecida
  [2026-06-01 15:00:00] Radiacao=6.2 mas severidade="normal"
    Evento: Operacoes normais retomadas
  [2026-06-02 13:00:00] Radiacao=6.2 mas severidade="normal"
    Evento: Geracao estavel
  [2026-06-03 12:00:00] Radiacao=6.2 mas severidade="normal"
    Evento: Pico energetico do dia
  [2026-06-03 13:00:00] Radiacao=6.5 mas severidade="normal"
    Evento: INCONSISTENCIA: radiacao alta mas sensores reportam normal
  [2026-06-03 16:00:00] Radiacao=7.5 mas severidade="normal"
    Evento: Radiacao em declinio
  [2026-06-04 12:00:00] Radiacao=6.2 mas severidade="normal"
    Evento: Relatorio de meio-dia: status nominal
  [2026-06-04 13:00:00] Radiacao=6.5 mas severidade="normal"
  

## 7. Resumo Final da Missão

In [ ]:
print('=' * 70)
print('RELATÓRIO FINAL - SISTEMA DE MONITORAMENTO ESPACIAL')
print('=' * 70)
print(f'Período analisado: {df["data_hora"].min()} a {df["data_hora"].max()}')
print(f'Total de registros: {len(df)}')
print(f'Eventos críticos: {(df["nivel_diagnostico"] == "CRITICO").sum()}')
print(f'Eventos em alerta: {(df["nivel_diagnostico"] == "ALERTA").sum()}')
print(f'Eventos normais: {(df["nivel_diagnostico"] == "NORMAL").sum()}')
print(f'Inconsistencias detectadas: {len(inconsistencias)}')
print(f'Previsao de reserva em 6h: {previsao_6h:.1f}%')

if b > 0:
    tendencia_energetica = "Positiva"
else:
    tendencia_energetica = "Negativa"
print(f'Tendencia energetica: {tendencia_energetica}')
print('=' * 70)

RELATORIO FINAL - SISTEMA DE MONITORAMENTO ESPACIAL
Periodo analisado: 2026-06-01 00:00:00 a 2026-06-04 23:00:00
Total de registros: 96
Eventos criticos: 3
Eventos em alerta: 19
Eventos normais: 74
Inconsistencias detectadas: 9
Previsao de reserva em 6h: 65.9%
Tendencia energetica: Positiva
